In [1]:
# Dependencies are installed in the demo-level Python 3.12 environment.

# Setup
You should have a local `.env` file with at least:
* `X_FUNCTIONS_KEY`
* `AZURE_FUNCTIONS_ENDPOINT`
* `AZURE_OPENAI_API_KEY`
* `AZURE_OPENAI_ENDPOINT`


In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

False

For now, we're using an API key. We configure the non-Azure OpenAI client to talk to the OpenAI/v1 API in Foundry.

In [3]:
import json
import os
import subprocess
from pathlib import Path
from openai import OpenAI

PROJECT_ENDPOINT = "https://ncus-prakharg-demo-2026.services.ai.azure.com/api/projects/ncus-prakharg-demo-2026"
NOTEBOOK_DIR = Path(os.environ["DEMO_NOTEBOOK_DIR"])

def azure_cli_token():
    completed = subprocess.run(
        [
            r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd", "account", "get-access-token",
            "--resource", "https://ai.azure.com",
            "--query", "accessToken",
            "--output", "tsv",
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    return completed.stdout.strip()

client = OpenAI(
    base_url=PROJECT_ENDPOINT.rstrip("/") + "/openai/v1/",
    api_key=azure_cli_token,
)
print("> Created Entra-authenticated OpenAI client.")


> Created Entra-authenticated OpenAI client.


In [4]:
# We'll create a "unique enough" identifier that lets us run this notebook
# multiple times and easily keep track of things each run creates.
import uuid

UNIQUE_ENOUGH_KEY = str(uuid.uuid4()).split("-")[0]
print(f"> Using unique enough key: {UNIQUE_ENOUGH_KEY}")

> Using unique enough key: 7d9aba45


In [5]:
DEVELOPER_PROMPT = """
You are an expert in arithmetic problem solving. Given a target number and a list of
numbers, your task is to combine all of the numbers exactly once using addition (+),
subtraction (-), multiplication (x), or division (/) to reach the target.

- You must use every number exactly once.
- Use parentheses as needed to control the order of operations.
- Return only a valid JSON object with the following exact syntax:
  {
    "expression": <the expression>,
    "result": <the result>
  }
- The expression should be a string representation of the mathematical expression.
- The result should be a string containing the integer result.
- If the exact target is not possible, return the closest valid result using all numbers.

# Example 1
target: 850
numbers: [100, 75, 50, 25, 6, 3]

Output:
{
  "expression": "((100 * 6) + (75 + 25) + (3 * 50))",
  "result": 850
}

# Example 2
target: 945
numbers: [25, 100, 9, 3, 6, 2]

Output:
{
  "expression": "((100 * 9) + (6 * 3) + (25 + 2))",
  "result": 945
}

# Example 3
target: 310
numbers: [7, 50, 75, 3, 8, 2]

Output:
{
  "expression": "((75 * 3) + (50 * 2) - (8 + 7))",
  "result": 310
}
""".strip()

# Baseline Evaluation
We're first going to assess how some models perform natively at this task given the above prompt.

In [6]:
# Upload our datasets for eval

with open(NOTEBOOK_DIR / "data" / "countdown_eval_25.jsonl", mode="rb") as f:
    eval_file = client.files.create(file=f, purpose="evals")
eval_file = client.files.wait_for_processing(eval_file.id)

print(f"> Uploaded eval file {eval_file.id}")

> Uploaded eval file file-4680c157b57940518a1f5bc5898c4fba


In [7]:
# We need to describe the source and schema for our files.
EVAL_DATA_SOURCE = {
    "item_schema": {
        "type": "object",
        "properties": {
            "target": {"type": "integer"},
            "nums": {"type": "array", "items": {"type": "integer"}},
        },
    },
    "include_sample_schema": True,
    "type": "custom",
}

In [8]:
# Our expected Response schema from the models. We'll use structured outputs
# so the model must output only JSON.
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "math_expression",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"},
                "result": {"type": "string"},
            },
            "required": ["expression", "result"],
            "additionalProperties": False,
        },
    },
}

## Endpoint Grader
Our endpoint grader points to a remote endpoint that hosts the grader function.

It has an API surface that's basically the Python grader, but via POST'ing JSON
to an endpoint.

We're using an Azure Function with key based auth, so each request will need
the `X-Functions-Key` header with our authentication key.

In [9]:
import os

URL = "https://prg-rft-countdown-ncus.azurewebsites.net/api/grader"
function_key = os.environ.get("X_FUNCTIONS_KEY")
if not function_key:
    raise RuntimeError("X_FUNCTIONS_KEY must be supplied through the process environment.")

ENDPOINT_GRADER = {
    "type": "endpoint",
    "name": "Remote Countdown Grader",
    "url": URL,
    "headers": {"X-Functions-Key": function_key},
    "pass_threshold": 5.0,
}
print(f"> Defined endpoint grader for {URL}")


> Defined endpoint grader for https://prg-rft-countdown-1002.azurewebsites.net/api/grader


Let's now make sure our Endpoint Grader is reachable. If it's not, there's no sense in
submitting an Eval or RFT job. We can POST some sample input and check if we get back
a score.

In [10]:
# Manually test the grader.
import requests

data = {
    "item": {"target": 47, "nums": [86, 22, 88, 72]},
    "sample": {
        "output_json": {
            "expression": "(72 + 22) / (88 - 86)",
            "result": 47,
        },
    },
}

result = requests.post(
    URL,
    headers={
        "Content-Type": "application/json",
        "X-Functions-Key": os.getenv("X_FUNCTIONS_KEY"),
    },
    json=data,
)

print(f"> Got HTTP {result.status_code}")
if result.status_code == 200:
    score = result.json()["score"]
    print(f"> Score is {score}.")
    if score != 5:
        raise RuntimeError("Uh oh...you might want to check your grader!")

> Got HTTP 200
> Score is 5.


For comparison, we'll use the same code but run as a Python grader.

In practice, this makes no sense...but it demonstrates how similar the two
graders are in shape.

The code is in [grader.py](./grader.py), so we can import the module and
use Python's native `inspect` module to dump it as source code into a
string we pass to the grader.

In [11]:
# Let's load an analagous Python Grader using the same code.
from pprint import pprint
import inspect
import grader

code = inspect.getsource(grader)

PYTHON_GRADER = {
    "type": "python",
    "name": "Python Countdown Grader",
    "source": code,
    "pass_threshold": 5.0,
}
print("> Defined Python grader")
pprint(PYTHON_GRADER)

> Defined Python grader
{'name': 'Python Countdown Grader',
 'pass_threshold': 5.0,
 'source': '"""\n'
           'Python Grader logic for OpenAI Evals\n'
           '\n'
           'Example from '
           'https://github.com/azure-ai-foundry/fine-tuning/blob/main/Demos/RFT_Countdown/demo_with_python_grader.ipynb\n'
           '"""\n'
           'from typing import Any, Dict\n'
           'import ast\n'
           'import json\n'
           'import logging\n'
           'import re\n'
           '\n'
           '\n'
           'def _eval(n):\n'
           '    if isinstance(n, ast.Constant):\n'
           '        return n.value\n'
           '\n'
           '    if isinstance(n, ast.BinOp) and type(n.op) in {\n'
           '        ast.Add: lambda a, b: a + b,\n'
           '        ast.Sub: lambda a, b: a - b,\n'
           '        ast.Mult: lambda a, b: a * b,\n'
           '        ast.Div: lambda a, b: a / b,\n'
           '        ast.FloorDiv: lambda a, b: a // b,\n'
        

Now we're ready to define our Eval.

In [12]:
# Create our Eval
baseline_eval = client.evals.create(
    name=f"endpoint-countdown-eval-{UNIQUE_ENOUGH_KEY}",
    data_source_config=EVAL_DATA_SOURCE,
    testing_criteria=[PYTHON_GRADER, ENDPOINT_GRADER],
)
print(f"> Created eval {baseline_eval.id}:")
print(baseline_eval.to_json())

> Created eval eval_165ca63c88e94829972b5403f684cb74:
{
  "id": "eval_165ca63c88e94829972b5403f684cb74",
  "created_at": 1790986365,
  "data_source_config": {
    "schema": {
      "item": {
        "type": "object",
        "properties": {
          "target": {
            "type": "integer"
          },
          "nums": {
            "type": "array",
            "items": {
              "type": "integer"
            }
          }
        }
      },
      "sample": {
        "type": "object",
        "properties": {
          "output_text": {
            "type": "string"
          }
        }
      }
    },
    "type": "custom",
    "item_schema": {},
    "include_sample_schema": true
  },
  "metadata": {},
  "name": "endpoint-countdown-eval-7d9aba45",
  "object": "eval",
  "testing_criteria": [
    {
      "name": "Python Countdown Grader",
      "source": "\"\"\"\nPython Grader logic for OpenAI Evals\n\nExample from https://github.com/azure-ai-foundry/fine-tuning/blob/main/Demos/RFT

C:\Data\AIP\fine-tuning\Demos\Agentic_RFT_PrivatePreview\.venv\Lib\site-packages\pydantic\main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['label_model']` - serialized value may not be as expected [field_name='type', input_value='endpoint', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `StringCheckGrader` - serialized value may not be as expected [field_name='testing_criteria', input_value=LabelModelGrader(input=No...='}, pass_threshold=5.0), input_type=LabelModelGrader])
  PydanticSerializationUnexpectedValue(Expected `TestingCriterionEvalGraderTextSimilarity` - serialized value may not be as expected [field_name='testing_criteria', input_value=LabelModelGrader(input=No...='}, pass_threshold=5.0), input_type=LabelModelGrader])
  PydanticSerializationUnexpectedValue(Expected `TestingCriterionEvalGraderPython` - serialized value may not be as expected [field_name='testing_criteria', input_value=LabelModel

In [13]:
# Define our Runs
USER_PROMPT = """
target: {{item.target}}
numbers: {{item.nums}}
""".strip()

RUNS = []
for model in ["gpt-4.1", "o4-mini"]:
    RUN_DATA_SOURCE = {
        "type": "completions",
        "model": model,
        "source": {"type": "file_id", "id": eval_file.id},
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "developer",
                    "content": {"type": "input_text", "text": DEVELOPER_PROMPT},
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": {"type": "input_text", "text": USER_PROMPT},
                },
            ],
        },
        "sampling_params": {
            "response_format": RESPONSE_SCHEMA,
        },
    }
    run = client.evals.runs.create(
        name=f"endpoint-countdown-eval-run-{model}-{UNIQUE_ENOUGH_KEY}",
        eval_id=baseline_eval.id,
        data_source=RUN_DATA_SOURCE,
    )
    print(f"> Created run {run.id}: {run.name}")
    RUNS.append(run)

print(f"> Created {len(RUNS)} runs.")

> Created run evalrun_fdc325e040564813a587fddc7b842557: endpoint-countdown-eval-run-gpt-4.1-7d9aba45


> Created run evalrun_3c549ee47a44499f8439bee96868916b: endpoint-countdown-eval-run-o4-mini-7d9aba45
> Created 2 runs.


# Reinforcement Fine Tuning
Now let's train a model to improve upon our out-of-box o4-mini experience.

In [14]:
# Upload training and validation data.
with open(NOTEBOOK_DIR / "data" / "countdown_train_25.jsonl", mode="rb") as f:
    training_file = client.files.create(file=f, purpose="fine-tune")
training_file = client.files.wait_for_processing(training_file.id)
print(f"> Uploaded training file {training_file.id}")

with open(NOTEBOOK_DIR / "data" / "countdown_valid_10.jsonl", mode="rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")
validation_file = client.files.wait_for_processing(validation_file.id)
print(f"> Uploaded validation file {validation_file.id}")

> Uploaded training file file-890382083bcf4a34899c90d9d5af3b15


> Uploaded validation file file-8cf844b2545f49f180a13b67aa5292a0


We can drop the `pass_threshold` settings from the individual graders. We can set it on
the `multi_grader` we'll use to combine both Python and Endpoint graders.

In [15]:
import copy
import requests
import time

RFT_ENDPOINT_GRADER = copy.deepcopy(ENDPOINT_GRADER)
RFT_ENDPOINT_GRADER.pop("pass_threshold", None)
RFT_PYTHON_GRADER = copy.deepcopy(PYTHON_GRADER)
RFT_PYTHON_GRADER.pop("pass_threshold", None)

if len(RFT_PYTHON_GRADER["source"].strip()) < 100:
    raise RuntimeError("Python grader source is empty or unexpectedly short.")

schema = RESPONSE_SCHEMA["json_schema"]
if schema["name"] != "math_expression":
    raise RuntimeError("Response schema name must be math_expression.")
if schema["schema"]["properties"] != {
    "expression": {"type": "string"},
    "result": {"type": "string"},
}:
    raise RuntimeError("Response schema properties are not the concrete countdown schema.")

RFT_MULTI_GRADER = {
    "type": "multi",
    "name": "Combined local and remote grader",
    "graders": {
        "python": RFT_PYTHON_GRADER,
        "endpoint": RFT_ENDPOINT_GRADER,
    },
    "calculate_output": "(python + endpoint) / 2",
}

job_payload = {
    "suffix": f"ncus-countdown-payload-{UNIQUE_ENOUGH_KEY}",
    "model": "gpt-5-2025-08-07",
    "training_file": training_file.id,
    "validation_file": validation_file.id,
    "method": {
        "type": "reinforcement",
        "reinforcement": {
            "grader": RFT_MULTI_GRADER,
            "response_format": RESPONSE_SCHEMA,
        },
    },
}

response = requests.post(
    PROJECT_ENDPOINT.rstrip("/") + "/openai/v1/fine_tuning/jobs",
    headers={
        "Authorization": f"Bearer {azure_cli_token()}",
        "Content-Type": "application/json",
    },
    json=job_payload,
    timeout=120,
)
response.raise_for_status()
job_id = response.json()["id"]
job = client.fine_tuning.jobs.retrieve(job_id)
print(f"> Created RFT job {job.id}")
print(job.to_json())

while job.status not in {"succeeded", "failed", "cancelled"}:
    time.sleep(60)
    job = client.fine_tuning.jobs.retrieve(job.id)
    print(f"> RFT job {job.id}: {job.status}")

print("> Terminal RFT job:")
print(job.to_json())

> Created RFT job ftjob-0b889c0af906426d8d988c7df3270c06
{
  "id": "ftjob-0b889c0af906426d8d988c7df3270c06",
  "created_at": 1790986468,
  "model": "o4-mini-2025-04-16",
  "object": "fine_tuning.job",
  "seed": 1126900549,
  "status": "pending",
  "training_file": "file-890382083bcf4a34899c90d9d5af3b15",
  "validation_file": "file-8cf844b2545f49f180a13b67aa5292a0",
  "estimated_finish": 1790987788,
  "metadata": {
    "base_model": "o4-mini",
    "model_version": "2025-04-16",
    "created_by": "Prakhar Gupta",
    "created_by_email": "prakharg@microsoft.com"
  },
  "method": {
    "type": "reinforcement",
    "reinforcement": {
      "grader": {
        "name": "Combined local and remote grader",
        "type": "multi",
        "graders": {
          "python": {
            "type": "python",
            "image_tag": "2025-05-08",
            "source": "\"\"\"\nPython Grader logic for OpenAI Evals\n\nExample from https://github.com/azure-ai-foundry/fine-tuning/blob/main/Demos/RFT_Coun

C:\Data\AIP\fine-tuning\Demos\Agentic_RFT_PrivatePreview\.venv\Lib\site-packages\pydantic\main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `literal['string_check']` - serialized value may not be as expected [field_name='type', input_value='multi', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `TextSimilarityGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrader])
  PydanticSerializationUnexpectedValue(Expected `PythonGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrader])
  PydanticSerializationUnexpectedValue(Expected `ScoreModelGrader` - serialized value may not be as expected [field_name='grader', input_value=StringCheckGrader(input=N...python + endpoint) / 2'), input_type=StringCheckGrade

> RFT job ftjob-0b889c0af906426d8d988c7df3270c06: pending


> RFT job ftjob-0b889c0af906426d8d988c7df3270c06: failed
> Terminal RFT job:
{
  "id": "ftjob-0b889c0af906426d8d988c7df3270c06",
  "created_at": 1790986468,
  "error": {
    "code": "400",
    "message": "File preprocessing failed. For more details, please check the Logs tab.",
    "param": "training file"
  },
  "finished_at": 1790986606,
  "model": "o4-mini-2025-04-16",
  "object": "fine_tuning.job",
  "seed": 1126900549,
  "status": "failed",
  "training_file": "file-890382083bcf4a34899c90d9d5af3b15",
  "validation_file": "file-8cf844b2545f49f180a13b67aa5292a0",
  "estimated_finish": 1790987788,
  "metadata": {
    "base_model": "o4-mini",
    "model_version": "2025-04-16",
    "created_by": "Prakhar Gupta",
    "created_by_email": "prakharg@microsoft.com"
  },
  "method": {
    "type": "reinforcement",
    "reinforcement": {
      "grader": {
        "name": "Combined local and remote grader",
        "type": "multi",
        "graders": {
          "python": {
            "type": "